In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/eslamfawzysayed/aplication-4-4487/final_data_complete.xlsx


In [2]:
# ============================================================
# Step 1: Install libraries
# ============================================================

!pip install pymoo -q
!pip install openpyxl -q
!pip install gradio -q

print("✅ All libraries installed successfully")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 217.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 772.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.3/328.3 kB 885.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 840.8 kB/s eta 0:00:00
✅ All libraries installed successfully


In [3]:
import os
import pandas as pd

# ============================================================
# ✅ 
# ============================================================

# If there are any saved points
if os.path.exists('optimization_backup.csv'):
    df = pd.read_csv('optimization_backup.csv')
    count = len(df)
    print(f"✅ في {count} نقطة محفوظة")
    
    # اعمل checkpoint
    with open('checkpoint_idx.txt', 'w') as f:
        f.write(str(count))
    
    print(f"🔄 هتكمل من العينة رقم {count}")
    print("🚀 شغّل Cell 2 دلوقتي وهيكمل")
else:
    print("❌ مفيش نقط محفوظة")
    print("🔄 شغّل Cell 2 من الأول")

❌ مفيش نقط محفوظة
🔄 شغّل Cell 2 من الأول


In [4]:

# ============================================================

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.optimize import minimize
import joblib
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print("🌱 Hydrochar Optimization System - Full Run")
print("=" * 70)

# Load data
file_path = '/kaggle/input/datasets/eslamfawzysayed/aplication-4-4487/final_data_complete.xlsx'
df = pd.read_excel(file_path)
print(f"✅ Loaded {len(df)} rows of data")

# Define columns (as mentioned)
input_cols = [
    'C%', 'H%', 'N%', 'S%', 'O%', 'Ash%', 
    'Volatile matter%', 'Fixed carbon%', 
    'HHV (MJ/Kg)', 'DM(%)', 'water Content (%)',
    'Temprature(°C)', 'Reaction Time (min)'
]

output_cols = [
    'yield(%)', 'Cc (%)', 'Hc(%)', 'Nc(%)', 'Sc(%)', 'Oc(%)',
    'HHVc (MJ/Kg)', 'EY(%)', 'ED(%)', 'CR(%)',
    'HC- Volatile matter%', 'HC- Fixed carbon%', 'HC- Ash(%)'
]

# Base columns (feedstock properties only)
base_input_cols = [
    'C%', 'H%', 'N%', 'S%', 'O%', 'Ash%', 
    'Volatile matter%', 'Fixed carbon%', 
    'HHV (MJ/Kg)', 'DM(%)', 'water Content (%)'
]

# Filter existing columns
input_cols = [col for col in input_cols if col in df.columns]
output_cols = [col for col in output_cols if col in df.columns]
base_input_cols = [col for col in base_input_cols if col in df.columns]

print(f"📊 Inputs: {len(input_cols)} (11 properties + 2 conditions)")
print(f"📊 Outputs: {len(output_cols)} properties")
print(f"📊 Feedstock properties: {len(base_input_cols)} properties")

🌱 Hydrochar Optimization System - Full Run
✅ Loaded 271 rows of data
📊 Inputs: 13 (11 properties + 2 conditions)
📊 Outputs: 13 properties
📊 Feedstock properties: 11 properties


In [5]:
# ============================================================
# Step 3: Train forward models (for prediction during optimization)
# ============================================================

print("\n" + "=" * 70)
print("🔄 Training forward models (13 models)...")
print("=" * 70)

X = df[input_cols]  # 11 properties + temperature + time
Y = df[output_cols]  # 13 outputs

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

models_forward = {}

for i, output_col in enumerate(output_cols):
    print(f"  Training {i+1}/{len(output_cols)}: {output_col}")
    model = RandomForestRegressor(
        n_estimators=50,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, Y_train[output_col])
    models_forward[output_col] = model

print("✅ All forward models trained successfully")

# Min/max values for normalization
output_mins = {col: df[col].min() for col in output_cols}
output_maxs = {col: df[col].max() for col in output_cols}


🔄 Training forward models (13 models)...
  Training 1/13: yield(%)
  Training 2/13: Cc (%)
  Training 3/13: Hc(%)
  Training 4/13: Nc(%)
  Training 5/13: Sc(%)
  Training 6/13: Oc(%)
  Training 7/13: HHVc (MJ/Kg)
  Training 8/13: EY(%)
  Training 9/13: ED(%)
  Training 10/13: CR(%)
  Training 11/13: HC- Volatile matter%
  Training 12/13: HC- Fixed carbon%
  Training 13/13: HC- Ash(%)
✅ All forward models trained successfully


In [6]:
# ============================================================
# Step 4: Application criteria (according to the table)
# ============================================================

print("\n" + "=" * 70)
print("📊 Application criteria")
print("=" * 70)

application_criteria = {
    'Energy': {
        'name': 'Energy Production',
        'weights': {
            'yield(%)': 0.15, 'Cc (%)': 0.10, 'Hc(%)': 0.05,
            'Nc(%)': 0.05, 'Sc(%)': 0.05, 'Oc(%)': 0.05,
            'HHVc (MJ/Kg)': 0.25, 'EY(%)': 0.05, 'ED(%)': 0.05,
            'CR(%)': 0.05, 'HC- Volatile matter%': 0.05,
            'HC- Fixed carbon%': 0.10, 'HC- Ash(%)': 0.05
        },
        'targets': {
            'yield(%)': 'max', 'Cc (%)': 'max', 'Hc(%)': 'max',
            'Nc(%)': 'min', 'Sc(%)': 'min', 'Oc(%)': 'min',
            'HHVc (MJ/Kg)': 'max', 'EY(%)': 'max', 'ED(%)': 'max',
            'CR(%)': 'max', 'HC- Volatile matter%': 'min',
            'HC- Fixed carbon%': 'max', 'HC- Ash(%)': 'min'
        }
    },
    'Soil': {
        'name': 'Soil Amendment',
        'weights': {
            'yield(%)': 0.15, 'Nc(%)': 0.25, 'Sc(%)': 0.20,
            'HC- Ash(%)': 0.30, 'HHVc (MJ/Kg)': 0.10
        },
        'targets': {
            'yield(%)': 'max', 'Nc(%)': 'max', 'Sc(%)': 'max',
            'HC- Ash(%)': 'max', 'HHVc (MJ/Kg)': 'min'
        }
    },
    'Adsorbent': {
        'name': 'Adsorbent Material',
        'weights': {
            'yield(%)': 0.15, 'Nc(%)': 0.20, 'Sc(%)': 0.15,
            'Oc(%)': 0.20, 'HC- Ash(%)': 0.20, 'HHVc (MJ/Kg)': 0.10
        },
        'targets': {
            'yield(%)': 'max', 'Nc(%)': 'max', 'Sc(%)': 'max',
            'Oc(%)': 'max', 'HC- Ash(%)': 'max', 'HHVc (MJ/Kg)': 'min'
        }
    }
}

for app_name, criteria in application_criteria.items():
    print(f"\n🔹 {app_name}: {criteria['name']}")
    print(f"   Weights: {', '.join([f'{k}={v}' for k, v in criteria['weights'].items() if v > 0])}")


📊 Application criteria

🔹 Energy: Energy Production
   Weights: yield(%)=0.15, Cc (%)=0.1, Hc(%)=0.05, Nc(%)=0.05, Sc(%)=0.05, Oc(%)=0.05, HHVc (MJ/Kg)=0.25, EY(%)=0.05, ED(%)=0.05, CR(%)=0.05, HC- Volatile matter%=0.05, HC- Fixed carbon%=0.1, HC- Ash(%)=0.05

🔹 Soil: Soil Amendment
   Weights: yield(%)=0.15, Nc(%)=0.25, Sc(%)=0.2, HC- Ash(%)=0.3, HHVc (MJ/Kg)=0.1

🔹 Adsorbent: Adsorbent Material
   Weights: yield(%)=0.15, Nc(%)=0.2, Sc(%)=0.15, Oc(%)=0.2, HC- Ash(%)=0.2, HHVc (MJ/Kg)=0.1


In [7]:
# ============================================================
# Step 5: Define NSGA-II problem
# ============================================================

print("\n" + "=" * 70)
print("🔄 Defining NSGA-II problem...")
print("=" * 70)

class OptimizationProblem(Problem):
    def __init__(self, raw_material, models_forward, input_cols, output_cols, 
                 weights, targets, df, output_mins, output_maxs):
        
        self.raw_material = raw_material
        self.models_forward = models_forward
        self.output_cols = output_cols
        self.weights = weights
        self.targets = targets
        self.output_mins = output_mins
        self.output_maxs = output_maxs
        
        # Temperature and time bounds
        temp_min = df['Temprature(°C)'].min()
        temp_max = df['Temprature(°C)'].max()
        time_min = df['Reaction Time (min)'].min()
        time_max = df['Reaction Time (min)'].max()
        
        xl = np.array([temp_min, time_min])
        xu = np.array([temp_max, time_max])
        
        # Active objectives (with weight > 0)
        self.active_outputs = [col for col, w in weights.items() if w > 0]
        n_obj = len(self.active_outputs)
        
        super().__init__(
            n_var=2,
            n_obj=n_obj,
            xl=xl,
            xu=xu,
            elementwise=False
        )
    
    def _evaluate(self, X, out, *args, **kwargs):
        n_solutions = X.shape[0]
        
        # Repeat raw material for each solution
        X_input = np.tile(self.raw_material, (n_solutions, 1))
        X_input = np.column_stack([X_input, X[:, 0], X[:, 1]])
        
        # Predict all outputs
        predictions = {}
        for col in self.output_cols:
            predictions[col] = self.models_forward[col].predict(X_input)
        
        # Calculate objectives
        F = np.zeros((n_solutions, self.n_obj))
        
        for i, output_col in enumerate(self.active_outputs):
            pred = predictions[output_col]
            min_val = self.output_mins[output_col]
            max_val = self.output_maxs[output_col]
            weight = self.weights[output_col]
            
            # Normalize
            if max_val > min_val:
                norm_val = (pred - min_val) / (max_val - min_val + 1e-10)
            else:
                norm_val = 0.5
            
            # Maximize or minimize
            if self.targets.get(output_col) == 'min':
                F[:, i] = norm_val * weight
            else:
                F[:, i] = -norm_val * weight
        
        out["F"] = F

print("✅ NSGA-II problem defined successfully")


🔄 Defining NSGA-II problem...
✅ NSGA-II problem defined successfully


In [8]:
!pip install pymoo -q

In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score, mean_squared_error
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.optimize import minimize
import warnings
import os
import joblib
import pickle
import json
from datetime import datetime
import time
import threading
import signal
import sys
import shutil

warnings.filterwarnings('ignore')

# ============================================================
# 1️⃣ TOPSIS SELECTION METHOD
# ============================================================

def topsis_selection(pareto_front, weights=None):
    """
    TOPSIS Method for selecting best solution from Pareto front
    """
    n_solutions, n_objectives = pareto_front.shape
    
    if weights is None:
        weights = np.ones(n_objectives) / n_objectives
    else:
        weights = np.array(weights) / np.sum(weights)
    
    # Normalization
    normalized = np.zeros_like(pareto_front)
    for j in range(n_objectives):
        col = pareto_front[:, j]
        norm_factor = np.sqrt(np.sum(col**2))
        if norm_factor > 0:
            normalized[:, j] = col / norm_factor
        else:
            normalized[:, j] = col
    
    # Weighted matrix
    weighted = normalized * weights
    
    # Ideal and Nadir solutions
    ideal_solution = np.min(weighted, axis=0)
    nadir_solution = np.max(weighted, axis=0)
    
    # Distances
    dist_ideal = np.sqrt(np.sum((weighted - ideal_solution)**2, axis=1))
    dist_nadir = np.sqrt(np.sum((weighted - nadir_solution)**2, axis=1))
    
    # TOPSIS score
    eps = 1e-10
    topsis_scores = dist_nadir / (dist_ideal + dist_nadir + eps)
    
    best_idx = np.argmax(topsis_scores)
    
    return best_idx, topsis_scores

# ============================================================
# 2️⃣ ADVANCED CHECKPOINT SYSTEM
# ============================================================

class AdvancedCheckpoint:
    def __init__(self, checkpoint_dir='checkpoints'):
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(checkpoint_dir, exist_ok=True)
        self.metadata_file = os.path.join(checkpoint_dir, 'metadata.json')
        
    def save_checkpoint(self, data, iteration, **kwargs):
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        checkpoint_file = os.path.join(
            self.checkpoint_dir, 
            f'checkpoint_iter_{iteration}_{timestamp}.pkl'
        )
        
        checkpoint_data = {
            'data': data,
            'iteration': iteration,
            'timestamp': timestamp,
            'kwargs': kwargs
        }
        
        with open(checkpoint_file, 'wb') as f:
            pickle.dump(checkpoint_data, f)
        
        self._update_metadata(iteration, checkpoint_file)
        
        if isinstance(data, list) and len(data) > 0:
            pd.DataFrame(data).to_csv(
                os.path.join(self.checkpoint_dir, f'data_backup_iter_{iteration}.csv'),
                index=False
            )
        
        return checkpoint_file
    
    def _update_metadata(self, iteration, checkpoint_file):
        metadata = {}
        if os.path.exists(self.metadata_file):
            with open(self.metadata_file, 'r') as f:
                metadata = json.load(f)
        
        metadata[str(iteration)] = {
            'file': checkpoint_file,
            'timestamp': datetime.now().isoformat(),
            'iteration': iteration
        }
        
        with open(self.metadata_file, 'w') as f:
            json.dump(metadata, f, indent=2)
    
    def load_latest_checkpoint(self):
        if not os.path.exists(self.metadata_file):
            return None, 0
        
        with open(self.metadata_file, 'r') as f:
            metadata = json.load(f)
        
        if not metadata:
            return None, 0
        
        latest_iter = max(int(k) for k in metadata.keys())
        checkpoint_file = metadata[str(latest_iter)]['file']
        
        if os.path.exists(checkpoint_file):
            with open(checkpoint_file, 'rb') as f:
                checkpoint_data = pickle.load(f)
            return checkpoint_data['data'], latest_iter
        
        return None, 0

    def save_final_data(self, data, filename='final_optimal_data.csv'):
        if isinstance(data, list) and len(data) > 0:
            df = pd.DataFrame(data)
            df.to_csv(filename, index=False)
            df.to_excel(filename.replace('.csv', '.xlsx'), index=False)
            return df
        return None

# ============================================================
# 3️⃣ KAGGLE SESSION KEEPER
# ============================================================

class KaggleSessionKeeper:
    def __init__(self, interval=300):
        self.interval = interval
        self.running = True
        self.thread = threading.Thread(target=self._keep_alive, daemon=True)
        
    def start(self):
        self.running = True
        self.thread.start()
        
    def stop(self):
        self.running = False
        
    def _keep_alive(self):
        while self.running:
            time.sleep(self.interval)
            if self.running:
                if os.path.exists('final_optimal_data.csv'):
                    try:
                        temp_df = pd.read_csv('final_optimal_data.csv')
                        temp_df.to_csv('temp_keep_alive.csv', index=False)
                    except:
                        pass

# ============================================================
# 4️⃣ OPTIMIZATION PROBLEM CLASS (MODIFIED)
# ============================================================

class OptimizationProblemImproved(Problem):
    def __init__(self, raw_material, models_forward, input_cols, output_cols, 
                 weights, targets, df, output_mins, output_maxs):
        
        self.raw_material = raw_material
        self.models_forward = models_forward
        self.output_cols = output_cols
        self.weights = weights
        self.targets = targets
        self.output_mins = output_mins
        self.output_maxs = output_maxs
        
        temp_min = 170
        temp_max = 260
        time_min = 30
        time_max = 180
        
        xl = np.array([temp_min, time_min])
        xu = np.array([temp_max, time_max])
        
        self.active_outputs = [col for col, w in weights.items() if w > 0]
        n_obj = len(self.active_outputs)
        
        super().__init__(
            n_var=2,
            n_obj=n_obj,
            xl=xl,
            xu=xu,
            elementwise=False
        )
    
    def _evaluate(self, X, out, *args, **kwargs):
        n_solutions = X.shape[0]
        
        # بناء المصفوفة كاملة (11 ميزة أساسية + Temperature + Time)
        X_input = np.tile(self.raw_material, (n_solutions, 1))
        X_input = np.column_stack([X_input, X[:, 0], X[:, 1]])  # 11 + 2 = 13 ميزات
        
        predictions = {}
        for col in self.output_cols:
            if col in self.models_forward:
                predictions[col] = self.models_forward[col].predict(X_input)
            else:
                predictions[col] = np.zeros(n_solutions)
        
        F = np.zeros((n_solutions, self.n_obj))
        
        for i, output_col in enumerate(self.active_outputs):
            pred = predictions[output_col]
            min_val = self.output_mins.get(output_col, 0)
            max_val = self.output_maxs.get(output_col, 100)
            weight = self.weights[output_col]
            
            if max_val > min_val:
                norm_val = (pred - min_val) / (max_val - min_val + 1e-10)
            else:
                norm_val = 0.5
            
            # ✅ التعديل الأساسي: استخدام target لتحديد تعظيم أو تقليل
            if self.targets.get(output_col) == 'min':
                F[:, i] = -norm_val * weight  # سالبة للتقليل
            else:
                F[:, i] = norm_val * weight   # موجبة للتعظيم
        
        out["F"] = F

# ============================================================
# 5️⃣ GENERATE OPTIMAL DATA WITH TOPSIS + CHECKPOINT
# ============================================================

def generate_optimal_data_with_topsis(df, models_forward, input_cols, output_cols,
                                     output_mins, output_maxs, application_criteria,
                                     base_input_cols, checkpoint_dir='checkpoints',
                                     save_interval=3):
    """
    ✅ توليد البيانات المثالية مع TOPSIS ونظام الحفظ المتقدم
    """
    checkpoint_system = AdvancedCheckpoint(checkpoint_dir)
    
    all_data, start_iter = checkpoint_system.load_latest_checkpoint()
    if all_data is None:
        all_data = []
        start_iter = 0
    
    print(f"\n🔄 Starting from iteration: {start_iter}")
    print(f"📊 Current data points: {len(all_data)}")
    print(f"🎯 Selection method: TOPSIS")
    
    current_iter = start_iter
    
    try:
        for app_name, criteria in application_criteria.items():
            print(f"\n{'='*70}")
            print(f"📊 Generating data for application: {app_name}")
            print(f"{'='*70}")
            
            weights = criteria['weights']
            targets = criteria['targets']
            
            weights_list = [weights[col] for col in weights.keys() if weights[col] > 0]
            
            for idx in range(len(df)):
                current_iter += 1
                
                if current_iter <= start_iter:
                    continue
                
                try:
                    raw_material = [df.iloc[idx][col] for col in base_input_cols]
                    
                    problem = OptimizationProblemImproved(
                        raw_material, models_forward, input_cols, output_cols,
                        weights, targets, df, output_mins, output_maxs
                    )
                    
                    algorithm = NSGA2(
                        pop_size=80,
                        sampling=FloatRandomSampling(),
                        crossover=SBX(prob=0.9, eta=15),
                        mutation=PM(prob=0.1, eta=20),
                        eliminate_duplicates=True
                    )
                    
                    res = minimize(
                        problem,
                        algorithm,
                        ('n_gen', 150),
                        seed=42 + idx,
                        verbose=False
                    )
                    
                    if len(res.X) > 0:
                        pareto_front = res.F
                        
                        # 🔥 استخدام TOPSIS لاختيار الحل الأمثل
                        best_idx, topsis_scores = topsis_selection(
                            pareto_front, 
                            weights=weights_list
                        )
                        
                        best_x = res.X[best_idx]
                        
                        temp, time_val = best_x
                        temp = np.clip(temp, 170, 260)
                        time_val = np.clip(time_val, 30, 180)
                        
                        X_input = np.array(raw_material + [temp, time_val]).reshape(1, -1)
                        
                        predictions = {}
                        for col in output_cols:
                            if col in models_forward:
                                predictions[col] = models_forward[col].predict(X_input)[0]
                            else:
                                predictions[col] = 0
                        
                        row = {
                            'Application': app_name,
                            'Temperature': temp,
                            'Time': time_val,
                            'Original_Sample_ID': idx,
                            'Iteration_Number': current_iter,
                            'TOPSIS_Score': topsis_scores[best_idx]
                        }
                        
                        for i, col in enumerate(base_input_cols):
                            row[col] = raw_material[i]
                        
                        for col in output_cols:
                            row[f'Pred_{col}'] = predictions.get(col, 0)
                        
                        all_data.append(row)
                        
                        # حفظ كل save_interval نقطة
                        if len(all_data) % save_interval == 0:
                            checkpoint_system.save_checkpoint(
                                all_data, 
                                current_iter,
                                app=app_name,
                                samples_processed=idx+1
                            )
                            print(f"💾 Checkpoint saved: {len(all_data)} points at iteration {current_iter}")
                    
                    if (idx + 1) % 10 == 0:
                        print(f"   ✅ Progress: {idx+1}/{len(df)} - Total points: {len(all_data)}")
                        
                except Exception as e:
                    print(f"⚠️ خطأ في العينة {idx}: {e}")
                    checkpoint_system.save_checkpoint(
                        all_data, 
                        current_iter,
                        error=str(e),
                        app=app_name
                    )
                    continue
                
                if current_iter % 50 == 0:
                    checkpoint_system.save_checkpoint(
                        all_data, 
                        current_iter,
                        status='periodic_save'
                    )
                    print(f"⏰ Periodic checkpoint saved at iteration {current_iter}")
    
    except KeyboardInterrupt:
        print("\n⚠️ Process interrupted. Saving checkpoint...")
        checkpoint_system.save_checkpoint(all_data, current_iter, status='interrupted')
    except Exception as e:
        print(f"\n❌ Error: {e}. Saving emergency checkpoint...")
        checkpoint_system.save_checkpoint(all_data, current_iter, status='emergency', error=str(e))
        raise
    
    final_df = checkpoint_system.save_final_data(all_data)
    return final_df

# ============================================================
# 6️⃣ SPLIT DATA WITHOUT LEAKAGE
# ============================================================

def split_without_leakage(data, features_cols, target_cols, group_col='Original_Sample_ID', test_size=0.2):
    X = data[features_cols]
    y = data[target_cols]
    groups = data[group_col]
    
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))
    
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]
    
    print(f"\n📊 Data Split (No Leakage):")
    print(f"   Training: {len(X_train)} samples")
    print(f"   Testing: {len(X_test)} samples")
    
    return X_train, X_test, y_train, y_test

# ============================================================
# 7️⃣ TRAIN INVERSE MODEL
# ============================================================

def train_inverse_model(X_train, X_test, y_train, y_test):
    from sklearn.ensemble import RandomForestRegressor
    
    print("\n" + "=" * 70)
    print("🔄 TRAINING INVERSE MODEL")
    print("=" * 70)
    
    inverse_model = RandomForestRegressor(
        n_estimators=150,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
    
    inverse_model.fit(X_train, y_train)
    
    y_pred_train = inverse_model.predict(X_train)
    y_pred_test = inverse_model.predict(X_test)
    
    print(f"\n📊 Inverse Model Performance:")
    print(f"{'Target':<20} {'Train R²':<12} {'Test R²':<12}")
    print("-" * 50)
    
    for i, col in enumerate(y_train.columns):
        r2_train = r2_score(y_train.iloc[:, i], y_pred_train[:, i])
        r2_test = r2_score(y_test.iloc[:, i], y_pred_test[:, i])
        
        status = "✅" if r2_test >= 0.7 else "⚠️"
        print(f"{col:<20} {r2_train:.3f}       {r2_test:.3f}       {status}")
    
    print("=" * 70)
    return inverse_model

# ============================================================
# 8️⃣ SIGNAL HANDLER
# ============================================================

def signal_handler(sig, frame):
    print('\n🛑 Saving emergency backup...')
    if 'all_data' in locals():
        try:
            pd.DataFrame(all_data).to_csv('emergency_backup.csv', index=False)
        except:
            pass
    sys.exit(0)

signal.signal(signal.SIGINT, signal_handler)
signal.signal(signal.SIGTERM, signal_handler)

# ============================================================
# 🔟 MAIN EXECUTION - POLLUTANT ADSORBENT ONLY
# ============================================================

print("\n" + "=" * 70)
print("🚀 STARTING OPTIMIZATION WITH TOPSIS + CHECKPOINT")
print("🎯 TARGET: POLLUTANT ADSORBENT")
print("=" * 70)

# ============================================================
# 📂 تحميل الملف الأصلي من مسار Kaggle
# ============================================================

file_name = '/kaggle/input/datasets/eslamfawzysayed/aplication-4-4487/final_data_complete.xlsx'

print(f"\n📂 سيتم تحميل الملف: {file_name}")

# تحميل الملف
try:
    if file_name.endswith('.csv'):
        df = pd.read_csv(file_name)
    else:
        df = pd.read_excel(file_name)
    print(f"✅ تم تحميل {len(df)} صف و {len(df.columns)} عمود")
except Exception as e:
    print(f"❌ خطأ في تحميل الملف: {e}")
    sys.exit(1)

# ============================================================
# 📊 عرض أسماء الأعمدة
# ============================================================

print("\n" + "=" * 70)
print("📊 أسماء الأعمدة في DataFrame:")
print("=" * 70)
for i, col in enumerate(df.columns):
    print(f"  {i+1}. {col}")
print("=" * 70)

# ============================================================
# 📊 تعريف المتغيرات حسب أسماء الأعمدة
# ============================================================

# أعمدة المدخلات (المواد الخام)
base_input_cols = ['C%', 'H%', 'N%', 'S%', 'O%', 'Ash%', 'Volatile matter%', 'Fixed carbon%', 'HHV (MJ/Kg)', 'DM(%)', 'water Content (%)']

# تحقق من وجود الأعمدة
missing_cols = [col for col in base_input_cols if col not in df.columns]
if missing_cols:
    print(f"⚠️ الأعمدة التالية غير موجودة: {missing_cols}")
    print("📌 يرجى تحديث قائمة base_input_cols حسب أسماء الأعمدة الظاهرة أعلاه")

# أعمدة المخرجات (النتائج المتوقعة)
output_cols = ['yield(%)', 'Cc (%)', 'Hc(%)', 'Nc(%)', 'Sc(%)', 'Oc(%)', 'HHVc (MJ/Kg)', 'EY(%)', 'ED(%)', 'CR(%)', 'HC- Volatile matter%', 'HC- Fixed carbon%', 'HC- Ash(%)']

# الأعمدة المدخلة (نفس base_input_cols)
input_cols = base_input_cols.copy()

# حساب الحدود الدنيا والعليا للمخرجات
output_mins = {}
output_maxs = {}
for col in output_cols:
    if col in df.columns:
        output_mins[col] = df[col].min()
        output_maxs[col] = df[col].max()
    else:
        output_mins[col] = 0
        output_maxs[col] = 100

# ============================================================
# 🧠 تدريب النماذج الأمامية (FORWARD MODELS)
# ============================================================

print("\n" + "=" * 70)
print("🧠 TRAINING FORWARD MODELS")
print("=" * 70)

from sklearn.ensemble import RandomForestRegressor

models_forward = {}

# الميزات الكاملة (11 أساسية + Temperature + Time)
temp_col = 'Temprature(°C)' if 'Temprature(°C)' in df.columns else 'Temperature'
time_col = 'Reaction Time (min)' if 'Reaction Time (min)' in df.columns else 'Time'

full_features = base_input_cols + [temp_col, time_col]

# تحقق من وجود الأعمدة المطلوبة
available_features = [col for col in full_features if col in df.columns]
print(f"📊 عدد الميزات المتاحة للتدريب: {len(available_features)}")

# تحضير بيانات التدريب
X_train_forward = df[available_features].values

for col in output_cols:
    if col in df.columns:
        y_train_forward = df[col].values
        
        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=10,
            min_samples_split=5,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_train_forward, y_train_forward)
        models_forward[col] = model
        print(f"✅ تم تدريب النموذج لـ {col}")
    else:
        print(f"⚠️ العمود {col} غير موجود، سيتم تخطيه")

print("=" * 70)

# ============================================================
# ⚖️ تعريف معايير التطبيقات - POLLUTANT ADSORBENT
# ============================================================

print("\n" + "=" * 70)
print("⚖️ DEFINING POLLUTANT ADSORBENT CRITERIA")
print("=" * 70)

application_criteria = {
    'Pollutant_Adsorbent': {
        'weights': {
            'yield(%)': 0.15,          # ⬆ Maximize - وزن أقل
            'Nc(%)': 0.15,             # ⬆ Maximize
            'Sc(%)': 0.15,             # ⬆ Maximize
            'Oc(%)': 0.15,             # ⬆ Maximize
            'HC- Ash(%)': 0.20,        # ⬆ Maximize - أهم حاجة للامتزاز
            'Cc (%)': 0.05,            # ◯ Not important - وزن قليل
            'Hc(%)': 0.05,             # ◯ Not important - وزن قليل
            'HHVc (MJ/Kg)': 0.10       # ⬇ Minimize - عايزينها تقل
        },
        'targets': {
            'yield(%)': 'max',          # تعظيم
            'Nc(%)': 'max',             # تعظيم
            'Sc(%)': 'max',             # تعظيم
            'Oc(%)': 'max',             # تعظيم
            'HC- Ash(%)': 'max',        # تعظيم - أساسي
            'Cc (%)': 'max',            # تعظيم (بوزن قليل)
            'Hc(%)': 'max',             # تعظيم (بوزن قليل)
            'HHVc (MJ/Kg)': 'min'       # تقليل - عايزينها تقل
        }
    }
}

print("✅ تم تعريف معايير Pollutant Adsorbent:")
print("   📈 Maximize: yield, Nc, Sc, Oc, HC-Ash, Cc, Hc")
print("   📉 Minimize: HHVc")
print("   🎯 Priority: HC-Ash (20%) > Others (15%) > Cc/Hc (5%)")

# ============================================================
# 🗑️ DELETE OLD CHECKPOINTS
# ============================================================

checkpoint_dirs = ['optimization_checkpoints', 'checkpoints']

for dir_name in checkpoint_dirs:
    if os.path.exists(dir_name):
        shutil.rmtree(dir_name)
        print(f"✅ Deleted: {dir_name}")

for file in os.listdir('.'):
    if file.endswith('.pkl') and 'checkpoint' in file:
        os.remove(file)
        print(f"✅ Deleted: {file}")

print("=" * 70)
print("🚀 STARTING FRESH - POLLUTANT ADSORBENT ONLY FROM ITERATION 0")
print("=" * 70)

# بدء Session Keeper
try:
    session_keeper = KaggleSessionKeeper(interval=240)
    session_keeper.start()
    print("✅ Session keeper started")
except:
    print("⚠️ Session keeper not available")

# ============================================================
# 🚀 توليد POLLUTANT ADSORBENT فقط
# ============================================================

optimal_data = generate_optimal_data_with_topsis(
    df, models_forward, input_cols, output_cols,
    output_mins, output_maxs, 
    application_criteria,
    base_input_cols,
    checkpoint_dir='optimization_checkpoints',
    save_interval=3
)

# ============================================================
# 💾 حفظ النتائج النهائية
# ============================================================

if optimal_data is not None and len(optimal_data) > 0:
    # حفظ بأسماء مميزة للممتز
    optimal_data.to_csv('optimal_data_pollutant_adsorbent.csv', index=False)
    optimal_data.to_excel('optimal_data_pollutant_adsorbent.xlsx', index=False)
    print(f"\n🎉 تم الانتهاء! عدد النقاط: {len(optimal_data)}")
    print("✅ تم حفظ: optimal_data_pollutant_adsorbent.csv و optimal_data_pollutant_adsorbent.xlsx")
else:
    print("⚠️ No data generated!")

# ============================================================
# 🔄 تدريب النموذج العكسي
# ============================================================

if optimal_data is not None and len(optimal_data) > 0:
    feature_cols = base_input_cols + ['Application_Pollutant_Adsorbent']

    optimal_data_encoded = pd.get_dummies(optimal_data, columns=['Application'], prefix='Application')

    for col in feature_cols:
        if col not in optimal_data_encoded.columns:
            optimal_data_encoded[col] = 0

    target_cols = ['Temperature', 'Time'] + [col for col in optimal_data_encoded.columns if col.startswith('Pred_')]

    X_train, X_test, y_train, y_test = split_without_leakage(
        optimal_data_encoded, feature_cols, target_cols,
        group_col='Original_Sample_ID', test_size=0.2
    )

    inverse_model = train_inverse_model(X_train, X_test, y_train, y_test)

    joblib.dump(inverse_model, 'inverse_model_pollutant_adsorbent.pkl')
    print("✅ Inverse model saved to 'inverse_model_pollutant_adsorbent.pkl'")

# إيقاف Session Keeper
try:
    session_keeper.stop()
    print("⏹️ Session keeper stopped")
except:
    pass

print("\n" + "=" * 70)
print("🎉 OPTIMIZATION PIPELINE COMPLETED SUCCESSFULLY!")
print("📊 TARGET: POLLUTANT ADSORBENT")
print("=" * 70)


🚀 STARTING OPTIMIZATION WITH TOPSIS + CHECKPOINT
🎯 TARGET: POLLUTANT ADSORBENT

📂 سيتم تحميل الملف: /kaggle/input/datasets/eslamfawzysayed/aplication-4-4487/final_data_complete.xlsx
✅ تم تحميل 271 صف و 26 عمود

📊 أسماء الأعمدة في DataFrame:
  1. C%
  2. H%
  3. N%
  4. S%
  5. O%
  6. Ash%
  7. Volatile matter%
  8. Fixed carbon%
  9. HHV (MJ/Kg)
  10. DM(%)
  11. water Content (%)
  12. Temprature(°C)
  13. Reaction Time (min)
  14. yield(%)
  15. Cc (%)
  16. Hc(%)
  17. Nc(%)
  18. Sc(%)
  19. Oc(%)
  20. HHVc (MJ/Kg)
  21. EY(%)
  22. ED(%)
  23. CR(%)
  24. HC- Volatile matter%
  25. HC- Fixed carbon%
  26. HC- Ash(%)

🧠 TRAINING FORWARD MODELS
📊 عدد الميزات المتاحة للتدريب: 13
✅ تم تدريب النموذج لـ yield(%)
✅ تم تدريب النموذج لـ Cc (%)
✅ تم تدريب النموذج لـ Hc(%)
✅ تم تدريب النموذج لـ Nc(%)
✅ تم تدريب النموذج لـ Sc(%)
✅ تم تدريب النموذج لـ Oc(%)
✅ تم تدريب النموذج لـ HHVc (MJ/Kg)
✅ تم تدريب النموذج لـ EY(%)
✅ تم تدريب النموذج لـ ED(%)
✅ تم تدريب النموذج لـ CR(%)
✅ تم تدريب النمو

In [10]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score, mean_squared_error
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.optimize import minimize
import warnings
import os
import joblib
import pickle
import json
from datetime import datetime
import time
import threading
import signal
import sys
import shutil

warnings.filterwarnings('ignore')

# ============================================================
# 1️⃣ TOPSIS SELECTION METHOD
# ============================================================

def topsis_selection(pareto_front, weights=None):
    """
    TOPSIS Method for selecting best solution from Pareto front
    """
    n_solutions, n_objectives = pareto_front.shape
    
    if weights is None:
        weights = np.ones(n_objectives) / n_objectives
    else:
        weights = np.array(weights) / np.sum(weights)
    
    # Normalization
    normalized = np.zeros_like(pareto_front)
    for j in range(n_objectives):
        col = pareto_front[:, j]
        norm_factor = np.sqrt(np.sum(col**2))
        if norm_factor > 0:
            normalized[:, j] = col / norm_factor
        else:
            normalized[:, j] = col
    
    # Weighted matrix
    weighted = normalized * weights
    
    # Ideal and Nadir solutions
    ideal_solution = np.min(weighted, axis=0)
    nadir_solution = np.max(weighted, axis=0)
    
    # Distances
    dist_ideal = np.sqrt(np.sum((weighted - ideal_solution)**2, axis=1))
    dist_nadir = np.sqrt(np.sum((weighted - nadir_solution)**2, axis=1))
    
    # TOPSIS score
    eps = 1e-10
    topsis_scores = dist_nadir / (dist_ideal + dist_nadir + eps)
    
    best_idx = np.argmax(topsis_scores)
    
    return best_idx, topsis_scores

# ============================================================
# 2️⃣ ADVANCED CHECKPOINT SYSTEM
# ============================================================

class AdvancedCheckpoint:
    def __init__(self, checkpoint_dir='checkpoints'):
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(checkpoint_dir, exist_ok=True)
        self.metadata_file = os.path.join(checkpoint_dir, 'metadata.json')
        
    def save_checkpoint(self, data, iteration, **kwargs):
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        checkpoint_file = os.path.join(
            self.checkpoint_dir, 
            f'checkpoint_iter_{iteration}_{timestamp}.pkl'
        )
        
        checkpoint_data = {
            'data': data,
            'iteration': iteration,
            'timestamp': timestamp,
            'kwargs': kwargs
        }
        
        with open(checkpoint_file, 'wb') as f:
            pickle.dump(checkpoint_data, f)
        
        self._update_metadata(iteration, checkpoint_file)
        
        if isinstance(data, list) and len(data) > 0:
            pd.DataFrame(data).to_csv(
                os.path.join(self.checkpoint_dir, f'data_backup_iter_{iteration}.csv'),
                index=False
            )
        
        return checkpoint_file
    
    def _update_metadata(self, iteration, checkpoint_file):
        metadata = {}
        if os.path.exists(self.metadata_file):
            with open(self.metadata_file, 'r') as f:
                metadata = json.load(f)
        
        metadata[str(iteration)] = {
            'file': checkpoint_file,
            'timestamp': datetime.now().isoformat(),
            'iteration': iteration
        }
        
        with open(self.metadata_file, 'w') as f:
            json.dump(metadata, f, indent=2)
    
    def load_latest_checkpoint(self):
        if not os.path.exists(self.metadata_file):
            return None, 0
        
        with open(self.metadata_file, 'r') as f:
            metadata = json.load(f)
        
        if not metadata:
            return None, 0
        
        latest_iter = max(int(k) for k in metadata.keys())
        checkpoint_file = metadata[str(latest_iter)]['file']
        
        if os.path.exists(checkpoint_file):
            with open(checkpoint_file, 'rb') as f:
                checkpoint_data = pickle.load(f)
            return checkpoint_data['data'], latest_iter
        
        return None, 0

    def save_final_data(self, data, filename='final_optimal_data.csv'):
        if isinstance(data, list) and len(data) > 0:
            df = pd.DataFrame(data)
            df.to_csv(filename, index=False)
            df.to_excel(filename.replace('.csv', '.xlsx'), index=False)
            return df
        return None

# ============================================================
# 3️⃣ KAGGLE SESSION KEEPER
# ============================================================

class KaggleSessionKeeper:
    def __init__(self, interval=300):
        self.interval = interval
        self.running = True
        self.thread = threading.Thread(target=self._keep_alive, daemon=True)
        
    def start(self):
        self.running = True
        self.thread.start()
        
    def stop(self):
        self.running = False
        
    def _keep_alive(self):
        while self.running:
            time.sleep(self.interval)
            if self.running:
                if os.path.exists('final_optimal_data.csv'):
                    try:
                        temp_df = pd.read_csv('final_optimal_data.csv')
                        temp_df.to_csv('temp_keep_alive.csv', index=False)
                    except:
                        pass

# ============================================================
# 4️⃣ OPTIMIZATION PROBLEM CLASS
# ============================================================

class OptimizationProblemImproved(Problem):
    def __init__(self, raw_material, models_forward, input_cols, output_cols, 
                 weights, targets, df, output_mins, output_maxs):
        
        self.raw_material = raw_material
        self.models_forward = models_forward
        self.output_cols = output_cols
        self.weights = weights
        self.targets = targets
        self.output_mins = output_mins
        self.output_maxs = output_maxs
        
        temp_min = 170
        temp_max = 260
        time_min = 30
        time_max = 180
        
        xl = np.array([temp_min, time_min])
        xu = np.array([temp_max, time_max])
        
        self.active_outputs = [col for col, w in weights.items() if w > 0]
        n_obj = len(self.active_outputs)
        
        super().__init__(
            n_var=2,
            n_obj=n_obj,
            xl=xl,
            xu=xu,
            elementwise=False
        )
    
    def _evaluate(self, X, out, *args, **kwargs):
        n_solutions = X.shape[0]
        
        X_input = np.tile(self.raw_material, (n_solutions, 1))
        X_input = np.column_stack([X_input, X[:, 0], X[:, 1]])
        
        predictions = {}
        for col in self.output_cols:
            if col in self.models_forward:
                predictions[col] = self.models_forward[col].predict(X_input)
            else:
                predictions[col] = np.zeros(n_solutions)
        
        F = np.zeros((n_solutions, self.n_obj))
        
        for i, output_col in enumerate(self.active_outputs):
            pred = predictions[output_col]
            min_val = self.output_mins.get(output_col, 0)
            max_val = self.output_maxs.get(output_col, 100)
            weight = self.weights[output_col]
            
            if max_val > min_val:
                norm_val = (pred - min_val) / (max_val - min_val + 1e-10)
            else:
                norm_val = 0.5
            
            if self.targets.get(output_col) == 'min':
                F[:, i] = norm_val * weight
            else:
                F[:, i] = -norm_val * weight
        
        out["F"] = F

# ============================================================
# 5️⃣ GENERATE OPTIMAL DATA WITH TOPSIS + CHECKPOINT
# ============================================================

def generate_optimal_data_with_topsis(df, models_forward, input_cols, output_cols,
                                     output_mins, output_maxs, application_criteria,
                                     base_input_cols, checkpoint_dir='checkpoints',
                                     save_interval=3):
    """
    ✅ توليد البيانات المثالية مع TOPSIS ونظام الحفظ المتقدم
    """
    checkpoint_system = AdvancedCheckpoint(checkpoint_dir)
    
    all_data, start_iter = checkpoint_system.load_latest_checkpoint()
    if all_data is None:
        all_data = []
        start_iter = 0
    
    print(f"\n🔄 Starting from iteration: {start_iter}")
    print(f"📊 Current data points: {len(all_data)}")
    print(f"🎯 Selection method: TOPSIS")
    
    current_iter = start_iter
    
    try:
        for app_name, criteria in application_criteria.items():
            print(f"\n{'='*70}")
            print(f"📊 Generating data for application: {app_name}")
            print(f"{'='*70}")
            
            weights = criteria['weights']
            targets = criteria['targets']
            
            weights_list = [weights[col] for col in weights.keys() if weights[col] > 0]
            
            for idx in range(len(df)):
                current_iter += 1
                
                if current_iter <= start_iter:
                    continue
                
                try:
                    raw_material = [df.iloc[idx][col] for col in base_input_cols]
                    
                    problem = OptimizationProblemImproved(
                        raw_material, models_forward, input_cols, output_cols,
                        weights, targets, df, output_mins, output_maxs
                    )
                    
                    algorithm = NSGA2(
                        pop_size=80,
                        sampling=FloatRandomSampling(),
                        crossover=SBX(prob=0.9, eta=15),
                        mutation=PM(prob=0.1, eta=20),
                        eliminate_duplicates=True
                    )
                    
                    res = minimize(
                        problem,
                        algorithm,
                        ('n_gen', 150),
                        seed=42 + idx,
                        verbose=False
                    )
                    
                    if len(res.X) > 0:
                        pareto_front = res.F
                        
                        best_idx, topsis_scores = topsis_selection(
                            pareto_front, 
                            weights=weights_list
                        )
                        
                        best_x = res.X[best_idx]
                        
                        temp, time_val = best_x
                        temp = np.clip(temp, 170, 260)
                        time_val = np.clip(time_val, 30, 180)
                        
                        X_input = np.array(raw_material + [temp, time_val]).reshape(1, -1)
                        
                        predictions = {}
                        for col in output_cols:
                            if col in models_forward:
                                predictions[col] = models_forward[col].predict(X_input)[0]
                            else:
                                predictions[col] = 0
                        
                        row = {
                            'Application': app_name,
                            'Temperature': temp,
                            'Time': time_val,
                            'Original_Sample_ID': idx,
                            'Iteration_Number': current_iter,
                            'TOPSIS_Score': topsis_scores[best_idx]
                        }
                        
                        for i, col in enumerate(base_input_cols):
                            row[col] = raw_material[i]
                        
                        for col in output_cols:
                            row[f'Pred_{col}'] = predictions.get(col, 0)
                        
                        all_data.append(row)
                        
                        if len(all_data) % save_interval == 0:
                            checkpoint_system.save_checkpoint(
                                all_data, 
                                current_iter,
                                app=app_name,
                                samples_processed=idx+1
                            )
                            print(f"💾 Checkpoint saved: {len(all_data)} points at iteration {current_iter}")
                    
                    if (idx + 1) % 10 == 0:
                        print(f"   ✅ Progress: {idx+1}/{len(df)} - Total points: {len(all_data)}")
                        
                except Exception as e:
                    print(f"⚠️ خطأ في العينة {idx}: {e}")
                    checkpoint_system.save_checkpoint(
                        all_data, 
                        current_iter,
                        error=str(e),
                        app=app_name
                    )
                    continue
                
                if current_iter % 50 == 0:
                    checkpoint_system.save_checkpoint(
                        all_data, 
                        current_iter,
                        status='periodic_save'
                    )
                    print(f"⏰ Periodic checkpoint saved at iteration {current_iter}")
    
    except KeyboardInterrupt:
        print("\n⚠️ Process interrupted. Saving checkpoint...")
        checkpoint_system.save_checkpoint(all_data, current_iter, status='interrupted')
    except Exception as e:
        print(f"\n❌ Error: {e}. Saving emergency checkpoint...")
        checkpoint_system.save_checkpoint(all_data, current_iter, status='emergency', error=str(e))
        raise
    
    final_df = checkpoint_system.save_final_data(all_data)
    return final_df

# ============================================================
# 6️⃣ SPLIT DATA WITHOUT LEAKAGE
# ============================================================

def split_without_leakage(data, features_cols, target_cols, group_col='Original_Sample_ID', test_size=0.2):
    X = data[features_cols]
    y = data[target_cols]
    groups = data[group_col]
    
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))
    
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]
    
    print(f"\n📊 Data Split (No Leakage):")
    print(f"   Training: {len(X_train)} samples")
    print(f"   Testing: {len(X_test)} samples")
    
    return X_train, X_test, y_train, y_test

# ============================================================
# 7️⃣ TRAIN INVERSE MODEL
# ============================================================

def train_inverse_model(X_train, X_test, y_train, y_test):
    from sklearn.ensemble import RandomForestRegressor
    
    print("\n" + "=" * 70)
    print("🔄 TRAINING INVERSE MODEL")
    print("=" * 70)
    
    inverse_model = RandomForestRegressor(
        n_estimators=150,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
    
    inverse_model.fit(X_train, y_train)
    
    y_pred_train = inverse_model.predict(X_train)
    y_pred_test = inverse_model.predict(X_test)
    
    print(f"\n📊 Inverse Model Performance:")
    print(f"{'Target':<20} {'Train R²':<12} {'Test R²':<12}")
    print("-" * 50)
    
    for i, col in enumerate(y_train.columns):
        r2_train = r2_score(y_train.iloc[:, i], y_pred_train[:, i])
        r2_test = r2_score(y_test.iloc[:, i], y_pred_test[:, i])
        
        status = "✅" if r2_test >= 0.7 else "⚠️"
        print(f"{col:<20} {r2_train:.3f}       {r2_test:.3f}       {status}")
    
    print("=" * 70)
    return inverse_model

# ============================================================
# 8️⃣ SIGNAL HANDLER
# ============================================================

def signal_handler(sig, frame):
    print('\n🛑 Saving emergency backup...')
    if 'all_data' in locals():
        try:
            pd.DataFrame(all_data).to_csv('emergency_backup.csv', index=False)
        except:
            pass
    sys.exit(0)

signal.signal(signal.SIGINT, signal_handler)
signal.signal(signal.SIGTERM, signal_handler)

# ============================================================
# 🔟 MAIN EXECUTION - RESUME FROM CHECKPOINT
# ============================================================

print("\n" + "=" * 70)
print("🚀 RESUMING FROM CHECKPOINT - SOIL ONLY")
print("=" * 70)

# ============================================================
# 📂 تحميل الملف الأصلي
# ============================================================

file_name = '/kaggle/input/datasets/eslamfawzysayed/aplication-4-4487/final_data_complete.xlsx'

print(f"\n📂 سيتم تحميل الملف: {file_name}")

try:
    if file_name.endswith('.csv'):
        df = pd.read_csv(file_name)
    else:
        df = pd.read_excel(file_name)
    print(f"✅ تم تحميل {len(df)} صف و {len(df.columns)} عمود")
except Exception as e:
    print(f"❌ خطأ في تحميل الملف: {e}")
    sys.exit(1)

# ============================================================
# 📊 تعريف المتغيرات
# ============================================================

base_input_cols = ['C%', 'H%', 'N%', 'S%', 'O%', 'Ash%', 'Volatile matter%', 'Fixed carbon%', 'HHV (MJ/Kg)', 'DM(%)', 'water Content (%)']

output_cols = ['yield(%)', 'Cc (%)', 'Hc(%)', 'Nc(%)', 'Sc(%)', 'Oc(%)', 'HHVc (MJ/Kg)', 'EY(%)', 'ED(%)', 'CR(%)', 'HC- Volatile matter%', 'HC- Fixed carbon%', 'HC- Ash(%)']

input_cols = base_input_cols.copy()

output_mins = {}
output_maxs = {}
for col in output_cols:
    if col in df.columns:
        output_mins[col] = df[col].min()
        output_maxs[col] = df[col].max()
    else:
        output_mins[col] = 0
        output_maxs[col] = 100

# ============================================================
# 🧠 تدريب النماذج الأمامية
# ============================================================

print("\n" + "=" * 70)
print("🧠 TRAINING FORWARD MODELS")
print("=" * 70)

from sklearn.ensemble import RandomForestRegressor

models_forward = {}

temp_col = 'Temprature(°C)' if 'Temprature(°C)' in df.columns else 'Temperature'
time_col = 'Reaction Time (min)' if 'Reaction Time (min)' in df.columns else 'Time'

full_features = base_input_cols + [temp_col, time_col]
available_features = [col for col in full_features if col in df.columns]
print(f"📊 عدد الميزات المتاحة للتدريب: {len(available_features)}")

X_train_forward = df[available_features].values

for col in output_cols:
    if col in df.columns:
        y_train_forward = df[col].values
        
        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=10,
            min_samples_split=5,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_train_forward, y_train_forward)
        models_forward[col] = model
        print(f"✅ تم تدريب النموذج لـ {col}")

print("=" * 70)

# ============================================================
# ⚖️ تعريف معايير التطبيقات
# ============================================================

application_criteria = {
    'Soil': {
        'weights': {
            'yield(%)': 0.3,
            'Cc (%)': 0.2,
            'Hc(%)': 0.2,
            'HHVc (MJ/Kg)': 0.3
        },
        'targets': {
            'yield(%)': 'max',
            'Cc (%)': 'max',
            'Hc(%)': 'max',
            'HHVc (MJ/Kg)': 'max'
        }
    }
}

print("=" * 70)
print("🚀 RESUMING FROM CHECKPOINT - SOIL ONLY")
print("=" * 70)

# بدء Session Keeper
try:
    session_keeper = KaggleSessionKeeper(interval=240)
    session_keeper.start()
    print("✅ Session keeper started")
except:
    print("⚠️ Session keeper not available")

# ============================================================
# 🚀 استكمال التوليد من آخر checkpoint
# ============================================================

application_criteria_soil_only = {
    'Soil': application_criteria['Soil']
}

optimal_data = generate_optimal_data_with_topsis(
    df, models_forward, input_cols, output_cols,
    output_mins, output_maxs, 
    application_criteria_soil_only,
    base_input_cols,
    checkpoint_dir='optimization_checkpoints',
    save_interval=3
)

# ============================================================
# 💾 حفظ النتائج النهائية
# ============================================================

if optimal_data is not None and len(optimal_data) > 0:
    optimal_data.to_csv('optimal_data_final.csv', index=False)
    optimal_data.to_excel('optimal_data_final.xlsx', index=False)
    print(f"\n🎉 تم الانتهاء! عدد النقاط: {len(optimal_data)}")
    print("✅ تم حفظ: optimal_data_final.csv و optimal_data_final.xlsx")
else:
    print("⚠️ No data generated!")

# ============================================================
# 🔄 تدريب النموذج العكسي
# ============================================================

if optimal_data is not None and len(optimal_data) > 0:
    feature_cols = base_input_cols + ['Application_Energy', 'Application_Soil', 'Application_Adsorbent']

    optimal_data_encoded = pd.get_dummies(optimal_data, columns=['Application'], prefix='Application')

    for col in feature_cols:
        if col not in optimal_data_encoded.columns:
            optimal_data_encoded[col] = 0

    target_cols = ['Temperature', 'Time'] + [col for col in optimal_data_encoded.columns if col.startswith('Pred_')]

    X_train, X_test, y_train, y_test = split_without_leakage(
        optimal_data_encoded, feature_cols, target_cols,
        group_col='Original_Sample_ID', test_size=0.2
    )

    inverse_model = train_inverse_model(X_train, X_test, y_train, y_test)

    joblib.dump(inverse_model, 'inverse_model_topsis.pkl')
    print("✅ Inverse model saved to 'inverse_model_topsis.pkl'")

# إيقاف Session Keeper
try:
    session_keeper.stop()
    print("⏹️ Session keeper stopped")
except:
    pass

print("\n" + "=" * 70)
print("🎉 OPTIMIZATION PIPELINE COMPLETED SUCCESSFULLY!")
print("=" * 70)


🚀 RESUMING FROM CHECKPOINT - SOIL ONLY

📂 سيتم تحميل الملف: /kaggle/input/datasets/eslamfawzysayed/aplication-4-4487/final_data_complete.xlsx
✅ تم تحميل 271 صف و 26 عمود

🧠 TRAINING FORWARD MODELS
📊 عدد الميزات المتاحة للتدريب: 13
✅ تم تدريب النموذج لـ yield(%)
✅ تم تدريب النموذج لـ Cc (%)
✅ تم تدريب النموذج لـ Hc(%)
✅ تم تدريب النموذج لـ Nc(%)
✅ تم تدريب النموذج لـ Sc(%)
✅ تم تدريب النموذج لـ Oc(%)
✅ تم تدريب النموذج لـ HHVc (MJ/Kg)
✅ تم تدريب النموذج لـ EY(%)
✅ تم تدريب النموذج لـ ED(%)
✅ تم تدريب النموذج لـ CR(%)
✅ تم تدريب النموذج لـ HC- Volatile matter%
✅ تم تدريب النموذج لـ HC- Fixed carbon%
✅ تم تدريب النموذج لـ HC- Ash(%)
🚀 RESUMING FROM CHECKPOINT - SOIL ONLY
✅ Session keeper started

🔄 Starting from iteration: 270
📊 Current data points: 270
🎯 Selection method: TOPSIS

📊 Generating data for application: Soil
💾 Checkpoint saved: 273 points at iteration 273
💾 Checkpoint saved: 276 points at iteration 276
💾 Checkpoint saved: 279 points at iteration 279
   ✅ Progress: 10/271 - To